## Objective

Build a campaign-level performance dataset for all campaigns that
started and ended within the 2025 calendar year.

### Grain
One row per campaign.

### Key business questions

- How many customers participated?
- What transaction activity was generated?
- How much reward cost was incurred?
- How many new customers were acquired?
- Which campaigns performed best relative to their targets and costs?

### Important analytical distinction

Observed campaign transaction value is not equivalent to incremental
business impact. Incrementality will be analyzed separately in a later
notebook.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_scope AS

SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    team,
    objective,
    start_date,
    end_date,

    -- Campaign duration
    DATEDIFF(end_date, start_date) + 1 AS campaign_days,

    budget,
    target_customer_count,
    target_transaction_count,
    target_transaction_amount

FROM `campaign&promotion`.gold.dim_campaign
WHERE start_date >= DATE '2025-01-01'
  AND end_date <= DATE '2025-12-31';

In [0]:
%sql

SELECT
    COUNT(*) AS campaign_count
FROM campaign_scope;

campaign_count
64


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_participation AS

SELECT
    campaign_id,

    COUNT(DISTINCT CASE
        WHEN eligible_flag = TRUE
        THEN customer_id
    END) AS eligible_customers,

    COUNT(DISTINCT CASE
        WHEN participated_flag = TRUE
        THEN customer_id
    END) AS participating_customers

FROM `campaign&promotion`.gold.fct_campaign_participation
WHERE campaign_id IN (
    SELECT campaign_id
    FROM campaign_scope
)
GROUP BY campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_transactions AS

SELECT
    c.campaign_id,

    COUNT(DISTINCT t.transaction_id) AS transaction_count,

    COUNT(DISTINCT t.customer_id) AS transacting_customers,

    SUM(t.amount) AS transaction_value,

    SUM(t.fee) AS fee_revenue,

    SUM(t.cashback) AS transaction_cashback

FROM campaign_scope c

INNER JOIN `campaign&promotion`.gold.fct_campaign_participation p
    ON c.campaign_id = p.campaign_id
   AND p.participated_flag = TRUE

INNER JOIN `campaign&promotion`.gold.fct_transaction t
    ON p.customer_id = t.customer_id
   AND t.transaction_date BETWEEN c.start_date AND c.end_date
   AND t.status = 'SUCCESS'

GROUP BY c.campaign_id;

campaign_id,transaction_count,transacting_customers,transaction_value,fee_revenue,transaction_cashback
CMP000001,3504,1332,2.2650163E8,494964.0,4796306.0
CMP000002,4734,1146,3.1809166E8,753535.0,4164830.0
CMP000003,2063,537,1.3612714E8,299868.0,56418.0
CMP000004,1441,1012,8.132267E7,184914.0,44800.0
CMP000005,11771,2152,7.5159948E8,1763687.0,8900231.0
CMP000006,633,269,4.011899E7,91115.0,28771.0
CMP000007,11952,2502,7.8655763E8,1780911.0,551240.0
CMP000008,2274,996,1.4637113E8,309985.0,2383674.0
CMP000009,4605,1083,2.7144743E8,613815.0,0.0
CMP000010,1853,468,1.0752949E8,274253.0,697546.0


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_rewards AS

SELECT
    campaign_id,

    COUNT(DISTINCT redemption_id) AS redemption_count,

    COUNT(DISTINCT customer_id) AS rewarded_customers,

    SUM(reward_amount) AS reward_cost,

    SUM(cashback_amount) AS cashback_cost,

    SUM(discount_amount) AS discount_cost

FROM `campaign&promotion`.gold.fct_promotion_redemption

WHERE campaign_id IN (
    SELECT campaign_id
    FROM campaign_scope
)

GROUP BY campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_new_customers AS

SELECT
    c.campaign_id,

    COUNT(DISTINCT CASE
        WHEN d.registration_date BETWEEN c.start_date AND c.end_date
        THEN p.customer_id
    END) AS new_customers

FROM campaign_scope c

INNER JOIN `campaign&promotion`.gold.fct_campaign_participation p
    ON c.campaign_id = p.campaign_id
   AND p.participated_flag = TRUE

INNER JOIN `campaign&promotion`.gold.dim_customer d
    ON p.customer_id = d.customer_id

GROUP BY c.campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_performance AS

SELECT
    c.campaign_id,
    c.campaign_name,
    c.campaign_type,
    c.team,
    c.objective,
    c.start_date,
    c.end_date,
    c.campaign_days,

    -- Budget
    c.budget,

    -- Planned targets
    c.target_customer_count,
    c.target_transaction_count,
    c.target_transaction_amount,

    -- Participation
    COALESCE(p.eligible_customers, 0) AS eligible_customers,
    COALESCE(p.participating_customers, 0) AS participating_customers,

    ROUND(
        100.0 *
        COALESCE(p.participating_customers, 0)
        / NULLIF(p.eligible_customers, 0),
        2
    ) AS participation_rate_pct,

    -- Transactions
    COALESCE(t.transaction_count, 0) AS transaction_count,
    COALESCE(t.transacting_customers, 0) AS transacting_customers,
    COALESCE(t.transaction_value, 0) AS transaction_value,
    COALESCE(t.fee_revenue, 0) AS fee_revenue,

    ROUND(
        COALESCE(t.transaction_value, 0)
        / NULLIF(t.transaction_count, 0),
        2
    ) AS avg_transaction_value,

    -- Rewards
    COALESCE(r.redemption_count, 0) AS redemption_count,
    COALESCE(r.rewarded_customers, 0) AS rewarded_customers,
    COALESCE(r.reward_cost, 0) AS reward_cost,
    COALESCE(r.cashback_cost, 0) AS cashback_cost,
    COALESCE(r.discount_cost, 0) AS discount_cost,

    -- Acquisition
    COALESCE(n.new_customers, 0) AS new_customers,

    ROUND(
        COALESCE(r.reward_cost, 0)
        / NULLIF(n.new_customers, 0),
        2
    ) AS cac_reward_basis,

    -- Target achievement
    ROUND(
        100.0 *
        COALESCE(p.participating_customers, 0)
        / NULLIF(c.target_customer_count, 0),
        2
    ) AS customer_target_attainment_pct,

    ROUND(
        100.0 *
        COALESCE(t.transaction_count, 0)
        / NULLIF(c.target_transaction_count, 0),
        2
    ) AS transaction_target_attainment_pct,

    ROUND(
        100.0 *
        COALESCE(t.transaction_value, 0)
        / NULLIF(c.target_transaction_amount, 0),
        2
    ) AS transaction_value_target_attainment_pct,

    -- Observed efficiency, NOT true incremental ROI
    ROUND(
        COALESCE(t.transaction_value, 0)
        / NULLIF(r.reward_cost, 0),
        2
    ) AS transaction_value_to_reward_cost

FROM campaign_scope c

LEFT JOIN campaign_participation p
    ON c.campaign_id = p.campaign_id

LEFT JOIN campaign_transactions t
    ON c.campaign_id = t.campaign_id

LEFT JOIN campaign_rewards r
    ON c.campaign_id = r.campaign_id

LEFT JOIN campaign_new_customers n
    ON c.campaign_id = n.campaign_id;

In [0]:
%sql

SELECT *
FROM campaign_performance
ORDER BY transaction_value DESC;

campaign_id,campaign_name,campaign_type,team,objective,start_date,end_date,campaign_days,budget,target_customer_count,target_transaction_count,target_transaction_amount,eligible_customers,participating_customers,participation_rate_pct,transaction_count,transacting_customers,transaction_value,fee_revenue,avg_transaction_value,redemption_count,rewarded_customers,reward_cost,cashback_cost,discount_cost,new_customers,cac_reward_basis,customer_target_attainment_pct,transaction_target_attainment_pct,transaction_value_target_attainment_pct,transaction_value_to_reward_cost
CMP000049,First Transaction Cashback 6,Acquisition,A&G,Acquire new customers,2025-07-31,2025-09-20,52,1138000.0,5364,15512,3.92111E8,3864,2616,67.70,17370,2407,1.11038691E9,2551975.0,63925.56,2346,2346,1.2651942E7,3235942.0,9416000.0,2013,6285.12,48.77,111.98,283.18,87.76
CMP000039,Thingyan Cashback Festival 2,Seasonal,Marketing,Increase payment volume,2025-01-01,2025-03-02,61,1806000.0,6665,15460,3.53726E8,1349,957,70.94,14082,949,9.4281601E8,2172599.0,66951.85,937,937,4685000.0,0.0,4685000.0,0,null,14.36,91.09,266.54,201.24
CMP000046,Welcome Wallet Reward 2,Acquisition,A&G,Acquire new customers,2025-09-20,2025-11-12,54,2257000.0,9193,16550,7.2162E8,2969,1697,57.16,15538,1656,9.3749297E8,2089383.0,60335.5,1643,1643,2052731.0,0.0,1586000.0,1280,1603.7,18.46,93.89,129.92,456.71
CMP000058,Friend Referral Bonus 2,Referral,A&G,Drive referral registrations,2025-07-30,2025-09-15,48,1618000.0,4617,10485,2.32778E8,4548,1930,42.44,12900,1753,8.3453721E8,1874032.0,64692.81,1733,1733,9541167.0,9541167.0,0.0,1157,8246.47,41.80,123.03,358.51,87.47
CMP000007,Weekend Payment Bonus 2,Engagement,Marketing,Increase wallet usage,2025-09-09,2025-09-30,22,500000.0,1322,2110,5.0879E7,4306,2585,60.03,11952,2502,7.8655763E8,1780911.0,65809.71,2433,2433,1.5846E7,0.0,1.5846E7,0,null,195.54,566.45,1545.94,49.64
CMP000043,Daily Tap Rewards,Engagement,Marketing,Increase wallet usage,2025-06-20,2025-08-12,54,971000.0,3237,9487,3.31231E8,3268,1372,41.98,11264,1366,7.538814E8,1717904.0,66928.39,1365,1365,2612041.0,643830.0,1626000.0,0,null,42.38,118.73,227.6,288.62
CMP000005,Weekend Payment Bonus,Engagement,Marketing,Increase wallet usage,2025-11-27,2025-12-31,35,553000.0,1725,2568,1.05252E8,5362,2195,40.94,11771,2152,7.5159948E8,1763687.0,63851.8,2145,2145,7389307.0,6665356.0,0.0,0,null,127.25,458.37,714.1,101.71
CMP000027,Summer Shopping Promotion,Seasonal,Marketing,Increase payment volume,2025-04-10,2025-05-05,26,712000.0,1700,4490,8.6803E7,2281,1810,79.35,10714,1759,6.8907909E8,1582612.0,64315.76,1745,1745,3101419.0,1349419.0,1752000.0,0,null,106.47,238.62,793.84,222.18
CMP000044,First Transaction Cashback 5,Acquisition,Marketing,Acquire new customers,2025-09-23,2025-11-12,51,1455000.0,7263,23082,8.19349E8,2819,1201,42.60,10908,1142,6.5985431E8,1462894.0,60492.69,1139,1139,1997408.0,0.0,1677000.0,910,2194.95,16.54,47.26,80.53,330.36
CMP000045,Valentine Payment Delight 3,Seasonal,Marketing,Increase payment volume,2025-12-09,2025-12-31,23,925000.0,2626,6250,2.46207E8,5407,2818,52.12,8995,2559,5.6967281E8,1341374.0,63332.16,2441,2441,1.0850923E7,4895923.0,5955000.0,0,null,107.31,143.92,231.38,52.5


### Campaign Ranking

In [0]:
%sql

-- Highest Transaction Value

SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    transaction_value,
    reward_cost,
    participation_rate_pct
FROM campaign_performance
ORDER BY transaction_value DESC
LIMIT 10;

campaign_id,campaign_name,campaign_type,transaction_value,reward_cost,participation_rate_pct
CMP000049,First Transaction Cashback 6,Acquisition,1.11038691E9,1.2651942E7,67.70
CMP000039,Thingyan Cashback Festival 2,Seasonal,9.4281601E8,4685000.0,70.94
CMP000046,Welcome Wallet Reward 2,Acquisition,9.3749297E8,2052731.0,57.16
CMP000058,Friend Referral Bonus 2,Referral,8.3453721E8,9541167.0,42.44
CMP000007,Weekend Payment Bonus 2,Engagement,7.8655763E8,1.5846E7,60.03
CMP000043,Daily Tap Rewards,Engagement,7.538814E8,2612041.0,41.98
CMP000005,Weekend Payment Bonus,Engagement,7.5159948E8,7389307.0,40.94
CMP000027,Summer Shopping Promotion,Seasonal,6.8907909E8,3101419.0,79.35
CMP000044,First Transaction Cashback 5,Acquisition,6.5985431E8,1997408.0,42.60
CMP000045,Valentine Payment Delight 3,Seasonal,5.6967281E8,1.0850923E7,52.12


In [0]:
%sql
-- Highest Participation Rate
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    eligible_customers,
    participating_customers,
    participation_rate_pct
FROM campaign_performance
ORDER BY participation_rate_pct DESC
LIMIT 10;

campaign_id,campaign_name,campaign_type,eligible_customers,participating_customers,participation_rate_pct
CMP000027,Summer Shopping Promotion,Seasonal,2281,1810,79.35
CMP000006,Coffee Shop Cashback Week,Merchant Promotion,534,398,74.53
CMP000025,Year-End Wallet Bonanza,Seasonal,1444,1030,71.33
CMP000039,Thingyan Cashback Festival 2,Seasonal,1349,957,70.94
CMP000002,Valentine Payment Delight,Seasonal,1882,1305,69.34
CMP000049,First Transaction Cashback 6,Acquisition,3864,2616,67.70
CMP000038,Spend & Earn Challenge 4,Engagement,1690,1105,65.38
CMP000019,Frequent Payer Bonus,Transaction Growth,1505,940,62.46
CMP000036,Friend Referral Bonus,Referral,2298,1425,62.01
CMP000023,Spend & Earn Challenge 3,Engagement,1403,867,61.80


In [0]:
%sql
-- Best Observed Efficiency
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    transaction_value,
    reward_cost,
    transaction_value_to_reward_cost
FROM campaign_performance
WHERE reward_cost > 0
ORDER BY transaction_value_to_reward_cost DESC
LIMIT 10;

campaign_id,campaign_name,campaign_type,transaction_value,reward_cost,transaction_value_to_reward_cost
CMP000061,Bill Payment Reward 4,Transaction Growth,4.0126635E8,274955.0,1459.39
CMP000060,New User Cashback 4,Acquisition,3.499493E7,34241.0,1022.02
CMP000023,Spend & Earn Challenge 3,Engagement,4.9595315E8,789392.0,628.27
CMP000042,Welcome Wallet Reward,Acquisition,2.4983595E8,450590.0,554.46
CMP000046,Welcome Wallet Reward 2,Acquisition,9.3749297E8,2052731.0,456.71
CMP000044,First Transaction Cashback 5,Acquisition,6.5985431E8,1997408.0,330.36
CMP000043,Daily Tap Rewards,Engagement,7.538814E8,2612041.0,288.62
CMP000059,Merchant Dining Week 2,Merchant Promotion,1.806564E8,701662.0,257.47
CMP000004,Bill Payment Reward 2,Transaction Growth,8.132267E7,329973.0,246.45
CMP000053,Grocery Cashback Days 3,Merchant Promotion,1.6423567E8,696000.0,235.97


In [0]:
%sql
-- Highest new-customer volume
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    new_customers,
    cac_reward_basis
FROM campaign_performance
ORDER BY new_customers DESC
LIMIT 10;

campaign_id,campaign_name,campaign_type,new_customers,cac_reward_basis
CMP000049,First Transaction Cashback 6,Acquisition,2013,6285.12
CMP000055,New User Cashback 3,Acquisition,1364,2259.25
CMP000046,Welcome Wallet Reward 2,Acquisition,1280,1603.7
CMP000058,Friend Referral Bonus 2,Referral,1157,8246.47
CMP000044,First Transaction Cashback 5,Acquisition,910,2194.95
CMP000009,First Transaction Cashback,Acquisition,831,6645.01
CMP000021,Invite & Earn,Referral,724,8079.1
CMP000026,New User Cashback 2,Acquisition,660,9866.67
CMP000040,First Transaction Cashback 4,Acquisition,582,3800.53
CMP000048,Invite & Earn 2,Referral,500,9214.81


In [0]:
%sql

SELECT
    COUNT(*) AS campaign_count,
    COUNT(DISTINCT campaign_id) AS distinct_campaigns
FROM campaign_performance;

campaign_count,distinct_campaigns
64,64


In [0]:
%sql

SELECT
    campaign_id,
    COUNT(*) AS rows_per_campaign
FROM campaign_performance
GROUP BY campaign_id
HAVING COUNT(*) > 1;

campaign_id,rows_per_campaign
